# ROI-Based Segmentation Evaluation (Multi-Model)

This notebook evaluates segmentation metrics **within a Region of Interest (ROI)** derived from the ground truth annotations, comparing **all three model configurations** (3-channel, 4-channel, 7-channel) in a single run.

The ROI is created using morphological closing on the ground truth stone masks, which:
- Bridges gaps/joints between stones (keeping them in the evaluation)
- Excludes areas outside the annotated facade (removing out-of-bond detections from metrics)

## Workflow:
1. **Set parameters** (paths for GT and all 3 predictions, kernel size)
2. **Generate & visualize ROI** - adjust kernel size if needed
3. **Run evaluation** for all models within the ROI
4. **Compare results** across models

---
## Cell 1: Parameters

Set your paths and kernel size here. Re-run subsequent cells after changing these.

In [ ]:
# ============================================================
# PARAMETERS - EDIT THESE
# ============================================================

# Ground truth mask path
GT_MASK_PATH = "/path/to/ground_truth_mask.png"

# Prediction paths for all three models
PRED_PATHS = {
    '3-channel': "/path/to/3channel_prediction.png",   # Geometry-only (normal maps)
    '4-channel': "/path/to/4channel_prediction.png",   # Appearance-only (RGB + alpha)
    '7-channel': "/path/to/7channel_prediction.png"    # Combined (RGB + alpha + normals)
}

# Output directory for results
OUTPUT_DIR = "/path/to/output/"

# ROI Generation Parameters
# ---------------------------
# Kernel size for morphological closing (in pixels)
# This should be large enough to bridge gaps between stones.
# With gaps of 5-10cm and resolution of ~2-6 mm/px, gaps are roughly 8-60 pixels.
# A kernel of 40-60 pixels radius should work well. Adjust based on visualization.
KERNEL_RADIUS = 45  # Radius of the disk structuring element (diameter = 2*radius + 1)

# Class names (should match your color scheme)
CLASS_NAMES = ['Background', 'Ashlar', 'Polygonal', 'Quarry Stone']

# Model display colors for charts
MODEL_COLORS = {
    '3-channel': '#e74c3c',  # Red
    '4-channel': '#3498db',  # Blue
    '7-channel': '#2ecc71'   # Green
}

print(f"Ground Truth: {GT_MASK_PATH}")
print(f"\nPrediction Paths:")
for model_name, path in PRED_PATHS.items():
    print(f"  {model_name}: {path}")
print(f"\nOutput Dir: {OUTPUT_DIR}")
print(f"Kernel Radius: {KERNEL_RADIUS} pixels (diameter: {2*KERNEL_RADIUS + 1})")

---
## Cell 2: Imports and Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import seaborn as sns
from PIL import Image
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report
from scipy import ndimage
from skimage.morphology import disk, binary_closing
from typing import Dict, Tuple, List
import os
import warnings
warnings.filterwarnings('ignore')

Image.MAX_IMAGE_PIXELS = None

# RGB to class mapping
RGB_TO_CLASS = {
    (0, 0, 0): 0,       # Black -> Background
    (0, 0, 255): 1,     # Blue -> Ashlar
    (255, 0, 0): 2,     # Red -> Polygonal
    (255, 255, 0): 3    # Yellow -> Quarry Stone
}

CLASS_COLORS = ['#000000', '#0000FF', '#FF0000', '#FFFF00']

print("Imports complete.")

---
## Cell 3: Helper Functions

In [ ]:
def rgb_to_class_mask(rgb_image: np.ndarray, verbose: bool = True) -> np.ndarray:
    """
    Convert RGB mask to class indices.
    Handles unmapped colors by assigning to nearest class.
    """
    height, width = rgb_image.shape[:2]
    class_mask = np.zeros((height, width), dtype=np.uint8)
    
    for rgb_tuple, class_idx in RGB_TO_CLASS.items():
        color_mask = np.all(rgb_image == rgb_tuple, axis=2)
        class_mask[color_mask] = class_idx
    
    # Handle unmapped pixels (compression artifacts, anti-aliasing)
    mapped_pixels = np.zeros((height, width), dtype=bool)
    for rgb_tuple in RGB_TO_CLASS.keys():
        mapped_pixels |= np.all(rgb_image == rgb_tuple, axis=2)
    
    unmapped_count = np.sum(~mapped_pixels)
    if unmapped_count > 0 and verbose:
        print(f"    Note: {unmapped_count} pixels with unmapped colors -> mapped to nearest class")
        unmapped_indices = np.where(~mapped_pixels)
        for i in range(len(unmapped_indices[0])):
            y, x = unmapped_indices[0][i], unmapped_indices[1][i]
            pixel_rgb = rgb_image[y, x]
            min_dist = float('inf')
            nearest_class = 0
            for rgb_tuple, class_idx in RGB_TO_CLASS.items():
                dist = np.sqrt(np.sum((pixel_rgb.astype(float) - np.array(rgb_tuple).astype(float))**2))
                if dist < min_dist:
                    min_dist = dist
                    nearest_class = class_idx
            class_mask[y, x] = nearest_class
    
    return class_mask


def generate_roi_from_ground_truth(gt_class_mask: np.ndarray, kernel_radius: int) -> np.ndarray:
    """
    Generate ROI mask using morphological closing on stone classes.
    """
    stone_mask = (gt_class_mask > 0).astype(np.uint8)
    selem = disk(kernel_radius)
    roi_mask = binary_closing(stone_mask, selem)
    return roi_mask.astype(bool)


def calculate_metrics_within_roi(
    gt_mask: np.ndarray, 
    pred_mask: np.ndarray, 
    roi_mask: np.ndarray,
    class_names: List[str]
) -> Tuple[Dict[str, float], Dict[str, Dict[str, float]]]:
    """
    Calculate IoU and F1 scores only within the ROI.
    """
    num_classes = len(class_names)
    
    # Flatten and filter to ROI only
    roi_flat = roi_mask.flatten()
    gt_flat = gt_mask.flatten()[roi_flat]
    pred_flat = pred_mask.flatten()[roi_flat]
    
    # Calculate IoU per class
    iou_scores = {}
    for class_idx in range(num_classes):
        gt_binary = (gt_flat == class_idx)
        pred_binary = (pred_flat == class_idx)
        
        intersection = np.logical_and(gt_binary, pred_binary).sum()
        union = np.logical_or(gt_binary, pred_binary).sum()
        
        if union == 0:
            iou = np.nan  # Class not present
        else:
            iou = intersection / union
        
        iou_scores[class_names[class_idx]] = iou
    
    # Calculate F1 scores
    report = classification_report(
        gt_flat, pred_flat,
        labels=list(range(num_classes)),
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )
    
    f1_scores = {}
    for class_name in class_names:
        if class_name in report:
            f1_scores[class_name] = {
                'precision': report[class_name]['precision'],
                'recall': report[class_name]['recall'],
                'f1-score': report[class_name]['f1-score'],
                'support': report[class_name]['support']
            }
    f1_scores['macro_avg'] = report['macro avg']['f1-score']
    f1_scores['weighted_avg'] = report['weighted avg']['f1-score']
    
    return iou_scores, f1_scores


print("Helper functions defined.")

---
## Cell 4: Load Ground Truth and Generate ROI

In [ ]:
# Load ground truth
print("Loading ground truth mask...")
gt_rgb = np.array(Image.open(GT_MASK_PATH).convert('RGB'))
gt_mask = rgb_to_class_mask(gt_rgb)
print(f"  Shape: {gt_mask.shape}")
print(f"  Classes present: {np.unique(gt_mask)}")

# Generate ROI
print(f"\nGenerating ROI with kernel radius = {KERNEL_RADIUS}...")
roi_mask = generate_roi_from_ground_truth(gt_mask, KERNEL_RADIUS)

stone_pixels = np.sum(gt_mask > 0)
roi_pixels = np.sum(roi_mask)
print(f"  Stone pixels: {stone_pixels:,}")
print(f"  ROI pixels: {roi_pixels:,}")
print(f"  Gap pixels (bridged): {roi_pixels - stone_pixels:,}")

print("\n✓ Ground truth and ROI ready.")

---
## Cell 5: Load All Predictions

In [ ]:
# Load all prediction masks
pred_masks = {}

print("Loading prediction masks...")
for model_name, pred_path in PRED_PATHS.items():
    print(f"\n  {model_name}:")
    pred_rgb = np.array(Image.open(pred_path).convert('RGB'))
    pred_mask = rgb_to_class_mask(pred_rgb, verbose=True)
    
    # Verify shape matches
    assert pred_mask.shape == gt_mask.shape, f"Shape mismatch for {model_name}: {pred_mask.shape} vs GT {gt_mask.shape}"
    
    pred_masks[model_name] = pred_mask
    print(f"    Shape: {pred_mask.shape}")
    print(f"    Classes predicted: {np.unique(pred_mask)}")
    
    # Count out-of-ROI predictions
    out_of_roi = np.sum((pred_mask > 0) & ~roi_mask)
    print(f"    Out-of-ROI stone predictions: {out_of_roi:,} pixels")

print("\n✓ All predictions loaded.")

---
## Cell 6: Visualize ROI and All Predictions

In [ ]:
# Visualization of GT, ROI, and all predictions
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

cmap = ListedColormap(CLASS_COLORS)

# ROI boundary for overlay
roi_boundary = roi_mask.astype(float) - ndimage.binary_erosion(roi_mask).astype(float)

# Row 1: Ground Truth, ROI visualization, Stone mask
axes[0, 0].imshow(gt_mask, cmap=cmap, vmin=0, vmax=3)
axes[0, 0].contour(roi_boundary, colors='lime', linewidths=1.5)
axes[0, 0].set_title('Ground Truth + ROI Boundary', fontsize=12, fontweight='bold')
axes[0, 0].axis('off')

# Show bridged gaps
bridged_gaps = roi_mask & (gt_mask == 0)
axes[0, 1].imshow(gt_mask, cmap=cmap, vmin=0, vmax=3)
overlay = np.zeros((*gt_mask.shape, 4))
overlay[bridged_gaps] = [0, 1, 0, 0.5]
axes[0, 1].imshow(overlay)
axes[0, 1].set_title(f'ROI with Bridged Gaps (green)\nKernel r={KERNEL_RADIUS}', fontsize=12, fontweight='bold')
axes[0, 1].axis('off')

# Empty or legend
axes[0, 2].axis('off')
axes[0, 2].text(0.5, 0.5, f'ROI Statistics:\n\nTotal pixels: {roi_mask.size:,}\nROI pixels: {np.sum(roi_mask):,}\nStone pixels: {np.sum(gt_mask > 0):,}\nGap pixels: {np.sum(bridged_gaps):,}',
                ha='center', va='center', fontsize=12, transform=axes[0, 2].transAxes,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Row 2: All three predictions
for idx, (model_name, pred_mask) in enumerate(pred_masks.items()):
    axes[1, idx].imshow(pred_mask, cmap=cmap, vmin=0, vmax=3)
    axes[1, idx].contour(roi_boundary, colors='lime', linewidths=1.5)
    
    out_of_roi = np.sum((pred_mask > 0) & ~roi_mask)
    axes[1, idx].set_title(f'{model_name}\n({out_of_roi:,} px outside ROI)', fontsize=12, fontweight='bold')
    axes[1, idx].axis('off')

plt.tight_layout()
plt.show()

---
## Cell 7: Calculate Metrics for All Models

In [ ]:
# Calculate metrics for each model
all_results = {}

print("Calculating metrics within ROI for all models...\n")
print("="*70)

for model_name, pred_mask in pred_masks.items():
    print(f"\n{model_name.upper()}")
    print("-"*40)
    
    iou_scores, f1_scores = calculate_metrics_within_roi(gt_mask, pred_mask, roi_mask, CLASS_NAMES)
    
    # Calculate mean IoU (excluding background, handling NaN for absent classes)
    stone_iou_values = [v for k, v in iou_scores.items() if k != 'Background' and not np.isnan(v)]
    mean_iou_stones = np.mean(stone_iou_values) if stone_iou_values else 0.0
    
    all_results[model_name] = {
        'iou_scores': iou_scores,
        'f1_scores': f1_scores,
        'mean_iou_stones': mean_iou_stones
    }
    
    # Print per-class IoU
    print("  Per-Class IoU:")
    for class_name, iou in iou_scores.items():
        if np.isnan(iou):
            print(f"    {class_name:20s}: N/A (not present)")
        else:
            print(f"    {class_name:20s}: {iou:.4f}")
    print(f"  Mean IoU (stone classes): {mean_iou_stones:.4f}")
    print(f"  Macro F1: {f1_scores['macro_avg']:.4f}")

print("\n" + "="*70)
print("✓ All metrics calculated.")

---
## Cell 8: Comparative Summary Table

In [ ]:
# Build comparative summary table
summary_data = []

for model_name in PRED_PATHS.keys():
    results = all_results[model_name]
    row = {'Model': model_name}
    
    # Add IoU for each class
    for class_name in CLASS_NAMES:
        iou = results['iou_scores'].get(class_name, np.nan)
        row[f'IoU_{class_name}'] = iou
    
    # Add summary metrics
    row['Mean_IoU_Stones'] = results['mean_iou_stones']
    row['Macro_F1'] = results['f1_scores']['macro_avg']
    row['Weighted_F1'] = results['f1_scores']['weighted_avg']
    
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data)

# Display with formatting
print("\n" + "="*90)
print("COMPARATIVE SUMMARY (Within ROI)")
print("="*90)
print(summary_df.to_string(index=False, float_format=lambda x: f'{x:.4f}' if not np.isnan(x) else 'N/A'))
print("="*90)

# Highlight best model
best_model = summary_df.loc[summary_df['Mean_IoU_Stones'].idxmax(), 'Model']
best_iou = summary_df['Mean_IoU_Stones'].max()
print(f"\n→ Best performing model: {best_model} (Mean IoU = {best_iou:.4f})")

---
## Cell 9: Comparative IoU Bar Chart

In [ ]:
# Grouped bar chart comparing IoU across models
fig, ax = plt.subplots(figsize=(14, 7))

x = np.arange(len(CLASS_NAMES))
width = 0.25
multiplier = 0

for model_name in PRED_PATHS.keys():
    iou_values = [all_results[model_name]['iou_scores'][c] for c in CLASS_NAMES]
    # Replace NaN with 0 for plotting
    iou_values_plot = [v if not np.isnan(v) else 0 for v in iou_values]
    
    offset = width * multiplier
    bars = ax.bar(x + offset, iou_values_plot, width, 
                  label=model_name, color=MODEL_COLORS[model_name],
                  edgecolor='black', linewidth=1)
    
    # Add value labels
    for bar, val, orig_val in zip(bars, iou_values_plot, iou_values):
        if np.isnan(orig_val):
            label = 'N/A'
        else:
            label = f'{val:.3f}'
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                label, ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    multiplier += 1

ax.set_ylabel('IoU Score', fontsize=12)
ax.set_title(f'Per-Class IoU Comparison (Within ROI, Kernel r={KERNEL_RADIUS})', fontsize=14, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(CLASS_NAMES, fontsize=11)
ax.legend(loc='upper right', fontsize=11)
ax.set_ylim(0, 1.15)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
## Cell 10: Mean IoU Comparison

In [ ]:
# Bar chart for mean IoU (stone classes only)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

model_names = list(PRED_PATHS.keys())
mean_ious = [all_results[m]['mean_iou_stones'] for m in model_names]
macro_f1s = [all_results[m]['f1_scores']['macro_avg'] for m in model_names]
colors = [MODEL_COLORS[m] for m in model_names]

# Mean IoU
bars1 = axes[0].bar(model_names, mean_ious, color=colors, edgecolor='black', linewidth=1.5)
for bar, val in zip(bars1, mean_ious):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                 f'{val:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Mean IoU', fontsize=12)
axes[0].set_title('Mean IoU (Stone Classes)', fontsize=14, fontweight='bold')
axes[0].set_ylim(0, 1.1)
axes[0].grid(axis='y', alpha=0.3)

# Macro F1
bars2 = axes[1].bar(model_names, macro_f1s, color=colors, edgecolor='black', linewidth=1.5)
for bar, val in zip(bars2, macro_f1s):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                 f'{val:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=12)
axes[1].set_ylabel('Macro F1', fontsize=12)
axes[1].set_title('Macro F1 Score', fontsize=14, fontweight='bold')
axes[1].set_ylim(0, 1.1)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
## Cell 11: Save Results

In [ ]:
# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save summary table
summary_path = os.path.join(OUTPUT_DIR, 'roi_evaluation_summary.csv')
summary_df.to_csv(summary_path, index=False)
print(f"✓ Summary table saved to: {summary_path}")

# Save detailed results for each model
for model_name in PRED_PATHS.keys():
    results = all_results[model_name]
    
    # Build detailed report
    detail_data = []
    for class_name in CLASS_NAMES:
        row = {
            'Class': class_name,
            'IoU': results['iou_scores'].get(class_name, np.nan),
            'Precision': results['f1_scores'].get(class_name, {}).get('precision', np.nan),
            'Recall': results['f1_scores'].get(class_name, {}).get('recall', np.nan),
            'F1-Score': results['f1_scores'].get(class_name, {}).get('f1-score', np.nan),
            'Support': results['f1_scores'].get(class_name, {}).get('support', 0)
        }
        detail_data.append(row)
    
    detail_df = pd.DataFrame(detail_data)
    detail_path = os.path.join(OUTPUT_DIR, f'roi_evaluation_{model_name.replace("-", "_")}.csv')
    detail_df.to_csv(detail_path, index=False)
    print(f"✓ Detailed results for {model_name} saved to: {detail_path}")

# Save ROI mask
roi_path = os.path.join(OUTPUT_DIR, 'roi_mask.png')
Image.fromarray((roi_mask * 255).astype(np.uint8)).save(roi_path)
print(f"✓ ROI mask saved to: {roi_path}")

print("\n✓ All results saved.")

---
## Cell 12: Side-by-Side Prediction Comparison

In [ ]:
# Final side-by-side comparison (within ROI only)
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

cmap = ListedColormap(CLASS_COLORS)

# Ground truth
gt_in_roi = gt_mask.copy()
gt_in_roi[~roi_mask] = 0
axes[0].imshow(gt_in_roi, cmap=cmap, vmin=0, vmax=3)
axes[0].set_title('Ground Truth\n(within ROI)', fontsize=12, fontweight='bold')
axes[0].axis('off')

# Each model
for idx, (model_name, pred_mask) in enumerate(pred_masks.items()):
    pred_in_roi = pred_mask.copy()
    pred_in_roi[~roi_mask] = 0
    
    axes[idx + 1].imshow(pred_in_roi, cmap=cmap, vmin=0, vmax=3)
    mean_iou = all_results[model_name]['mean_iou_stones']
    axes[idx + 1].set_title(f'{model_name}\nMean IoU: {mean_iou:.4f}', fontsize=12, fontweight='bold')
    axes[idx + 1].axis('off')

plt.tight_layout()

# Save
comparison_path = os.path.join(OUTPUT_DIR, 'model_comparison.png')
plt.savefig(comparison_path, dpi=150, bbox_inches='tight')
print(f"✓ Comparison figure saved to: {comparison_path}")

plt.show()